In [ ]:
# Minimal, focused imports for the lesson (assume pyreadstat is installed)
from pprint import pprint

import numpy as np
import pyreadstat
from scipy import stats


In [ ]:
# Load the SPSS .sav file and keep the dataframe + metadata for later use.
# Path is relative to the notebook's location.
data_path = '../../data/0_raw/ZM_LFS_DATASET2024_Annual_10percent.sav'
print('Loading data from:', data_path)
df, metadata = pyreadstat.read_sav(data_path)

# %% [Select columns of interest]
# The original lesson selected a long list of human-readable question labels
# and mapped them back to the underlying column names. We keep that mapping
# but focus only on the existing mapping code here (no change to the chosen labels).
cols = [
    # --- Demographics & Background ---
    "Is ... Male or Female?",
    "How old was ... at (his/her) last birthday?",
    "What is the highest grade/level of education that ... has successfully completed?",
    "What is ...'s current marital status?",
    "What is ...'s relationship to the head of the household?",
    "1. Province",
    "2. District",

    # --- Employment & Work ---
    "In the main job/business that (NAME) has, is she/he...",
    "INDUSTRY",
    "Occupation",
    "How many hours does (NAME) usually work per week in his/her...? Main job",
    "How many hours does (NAME) usually work per week in his/her...? OVERALL TOTAL",
    "What is the frequency of .....'s income/earnings in his/her main job?",
    "Would (NAME) want to work more hours per week than usually worked, provided the extra hours are paid?",
    "Is ?. employed on the basis of a written contract or an oral agreement?",

    # --- Income & Earnings ---
    "What is your annually/monthly/weekly/daily/hourly wage or salary before deductions?",
    "What are your annual/monthly/weekly/daily/hourly earnings after expenses?",
    "At what age did NAME start work for the first time in his /her life",

    # --- Time Use: Household Activities ---
    "During the last 7 days how much time did  (NAME) spend on Cleaning the house, washing clothes, cooking or shopping for the household",
    "During the last 7 days how much time did  (NAME) spend on Fetching water from natural or public sources for use by the household",
    "During the last 7 days how much time did (NAME) spend on Collecting firewood or other natural products for use as fuel by the household",
    "In the last 7 days, how much time did (NAME) spend on Leisure e.g., playing sports, watching TV etc.?",
    "In the last 7 days, how much time did (NAME) spend on Personal care e.g bathing, eating and sleeping?",
    "In the last 7 days how much time did name spend travelling from home to\xa0place\xa0of\xa0work",

    # --- Time Use: Hours by Day of Week (Main Job) ---
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Monday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Tuesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Wednesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Thursday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Friday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Saturday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Sunday Main job?",

    # --- Financial Inclusion ---
    "P.20. Do you own a mobile phone",
    "P.21. Do you have a mobile money account in your own name",
    "P.23. How often do you use mobile money?",
    "P.25.  On a scale of 1 to 4, Do you find mobile money services to be cheap or expensive?",
    "P.30.A Savings at a bank",
    "P.30.G.Savings with savings group",
    "P.30.D.Savings that you keep on your mobile phone",
    "What method do you mainly use to pay for food/groceries?",

    # --- Education ---
    "Can... read and write in any language?",
    "Has... ever attended school?",
    "Is (NAME) currently attending school?",
    "Have (NAME) ever repeated any level of schooling any point in time?",
    "At what age did (NAME) begin school?",
]

names_to_labels = metadata.column_names_to_labels
names_to_labels_reduced = {}
selected_names = []
for label in cols:
    for colname, collabel in names_to_labels.items():
        if collabel == label:
            selected_names.append(colname)
            names_to_labels_reduced[colname] = collabel

print('Mapped labels -> column names:')
pprint(names_to_labels_reduced)

# Keep only the selected columns to keep examples manageable
df = df[selected_names]

# Keep value labels handy
variable_value_labels = metadata.variable_value_labels


In [ ]:
# Small, simple helper for lesson use (uses scipy.stats.describe)
def descriptive_report(data, name="Variable"):
    """Print a compact descriptive statistics report for a numeric array/series.
    Uses nan-safe describe and trim mean for a robust example.
    """
    arr = np.asarray(data)
    desc = stats.describe(arr, nan_policy='omit')

    print(f"=== Descriptive Report: {name} ===")
    print(f"N:               {desc.nobs}")
    print(f"Mean:            {desc.mean:,.2f}")
    print(f"5% Trimmed Mean: {stats.trim_mean(arr, 0.05):,.2f}")
    print(f"Median:          {np.median(arr):,.2f}")
    print(f"Std Dev:         {np.sqrt(desc.variance):,.2f}")
    print(f"Min:             {desc.minmax[0]:,.2f}")
    print(f"Max:             {desc.minmax[1]:,.2f}")
    print(f"Skewness:        {desc.skewness:.3f}")
    print(f"Excess Kurtosis: {desc.kurtosis:.3f}")
    print(f"Std Error:       {stats.sem(arr):,.2f}")

# Example: show descriptive stats for FA3 (age) where available
print('\nDescriptive report for FA3 (age)')
descriptive_report(df[df['FA3'].notna()]['FA3'], name='Age (FA3)')


In [ ]:
# Demonstrate how an extreme outlier affects z-scores and IQR-based fences.
# We'll show the data 'before' (including the sentinel extreme value) and 'after'
# (with the sentinel removed), print counts, and plot histograms + boxplots.

# Build the wages/age table and drop true missing values
df_wages_all = df[['FA1', 'FA3']].dropna()
# Keep only respondents with FA1 == 1 to match the original lesson filter
df_wages_all = df_wages_all[df_wages_all['FA1'] == 1]

# Define the sentinel/extreme value used in the original data
sentinel = 1508843.0

# BEFORE: include the sentinel (extreme outlier)
df_before = df_wages_all.copy()

# AFTER: remove the sentinel value to see how diagnostics change
df_after = df_before[df_before['FA3'] != sentinel]

# Compute z-scores and count how many observations exceed |z| > 3
zs_before = stats.zscore(df_before['FA3'].to_numpy())
zs_after = stats.zscore(df_after['FA3'].to_numpy())

count_z_before = np.sum(np.abs(zs_before) > 3)
count_z_after = np.sum(np.abs(zs_after) > 3)

# IQR-based fences (use the 3*IQR multiplier as in the original notebook example)
iqr_before = stats.iqr(df_before['FA3'].to_numpy())
iqr_after = stats.iqr(df_after['FA3'].to_numpy())
q1_before = np.percentile(df_before['FA3'], 25)
q3_before = np.percentile(df_before['FA3'], 75)
q1_after = np.percentile(df_after['FA3'], 25)
q3_after = np.percentile(df_after['FA3'], 75)

lower_fence_before = q1_before - 3 * iqr_before
upper_fence_before = q3_before + 3 * iqr_before
lower_fence_after = q1_after - 3 * iqr_after
upper_fence_after = q3_after + 3 * iqr_after

extreme_iqr_before = df_before[(df_before['FA3'] < lower_fence_before) | (df_before['FA3'] > upper_fence_before)]
extreme_iqr_after = df_after[(df_after['FA3'] < lower_fence_after) | (df_after['FA3'] > upper_fence_after)]

print('\nSummary of outlier diagnostics:')
print(f'  Total rows (before): {len(df_before)}, (after): {len(df_after)}')
print(f'  Z-score outliers (|z|>3) before: {count_z_before}, after: {count_z_after}')
print(f'  IQR extreme rows before: {len(extreme_iqr_before)}, after: {len(extreme_iqr_after)}')
print(f'  IQR fences before: lower={lower_fence_before:.2f}, upper={upper_fence_before:.2f}')
print(f'  IQR fences after:  lower={lower_fence_after:.2f}, upper={upper_fence_after:.2f}')

# Show the sentinel row (if present) and its z-score in the 'before' data
sentinel_rows = df_before[df_before['FA3'] == sentinel]
if len(sentinel_rows) > 0:
    print('\nSentinel row(s) found (first row shown):')
    print(sentinel_rows.head(1))
    # Find the z-score for sentinel (take first occurrence)
    sentinel_index = sentinel_rows.index[0]
    sentinel_pos = df_before.index.get_loc(sentinel_index)
    print('Sentinel z-score (before):', zs_before[sentinel_pos])
else:
    print('\nNo sentinel value found in the before dataset.')

# Plot before vs after: histograms and boxplots side-by-side for easy comparison
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Histogram - before
    axes[0, 0].hist(df_before['FA3'], bins=30, color='C0', alpha=0.6)
    axes[0, 0].set_title('Histogram FA3 - BEFORE (includes sentinel outlier)')
    axes[0, 0].axvline(lower_fence_before, color='r', linestyle='--', label='IQR fences')
    axes[0, 0].axvline(upper_fence_before, color='r', linestyle='--')
    axes[0, 0].legend()

    # Boxplot - before
    axes[1, 0].boxplot(df_before['FA3'].values, vert=False)
    axes[1, 0].set_title('Boxplot FA3 - BEFORE')

    # Histogram - after
    axes[0, 1].hist(df_after['FA3'], bins=30, color='C2', alpha=0.6)
    axes[0, 1].set_title('Histogram FA3 - AFTER (sentinel removed)')
    axes[0, 1].axvline(lower_fence_after, color='r', linestyle='--', label='IQR fences')
    axes[0, 1].axvline(upper_fence_after, color='r', linestyle='--')
    axes[0, 1].legend()

    # Boxplot - after
    axes[1, 1].boxplot(df_after['FA3'].values, vert=False)
    axes[1, 1].set_title('Boxplot FA3 - AFTER')

    plt.tight_layout()
    plt.show()
except Exception:
    print('\nMatplotlib not available or running headless; skipping plots.')

# End of wage/age outlier demonstration


In [ ]:
# Show the value-label mapping for FA1 (if available)
print('\nValue labels for FA1 (if present):')
print(variable_value_labels.get('FA1', 'No value labels found for FA1'))

# Simple mean example for A3 (age)
print('\nMean of A3 (full column, NaNs ignored):', df['A3'].mean())


In [ ]:
# Take a moderate-size random sample from A3 to demonstrate a one-sample t-test
sample_data = df[df['A3'] > 5].sample(100, random_state=42)['A3'].to_numpy()
population_mean = df['A3'].mean()

t_stat, p_value = stats.ttest_1samp(sample_data, popmean=population_mean)
print('\nOne-sample t-test vs LFS mean:')
print(f'Sample mean:   {np.mean(sample_data):,.2f}')
print(f'T-statistic:   {t_stat:.4f}')
print(f'P-value:       {p_value:.6f}')

if p_value < 0.05:
    print('→ Reject H₀: Sample mean is significantly different from the LFS mean (alpha=0.05)')
else:
    print('→ Fail to reject H₀: No significant difference from LFS mean')


In [ ]:
# Repeat a small simulation to see how often a random sample mean from n=100
# exceeds the observed sample mean; keep iterations modest for the lesson.
res = []
iterations = 2000
obs_mean = np.mean(sample_data)
for _ in range(iterations):
    smean = np.mean(df.sample(100)['A3'].to_numpy())
    if smean > obs_mean:
        res.append(smean)
pct = len(res) / iterations * 100
print(f'\nIn {iterations} random samples of size 100, {pct:.2f}% had mean > observed sample mean')
